# Payments: the fine-tuned model with the new refund-amount rule

Reruns the fine-tuned model you trained earlier, with every rule on, including the new one: a refund must carry the amount the merchant stated. No training. About 40 minutes.

1. **Runtime > Change runtime type > T4 GPU**, save.
2. **Runtime > Run all**.
3. When asked, upload the two files you saved earlier: `adapter_model.safetensors` and `adapter_config.json`.
4. Keep the tab open until `payments_rule_rerun.zip` downloads, then send it to Claude.

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 -b v2-split https://github.com/eshwarpotturi/slm-opti /content/slm-opti

In [ ]:
import os, sys, json, time, glob, zipfile
os.environ["TZ"] = "Asia/Kolkata"; time.tzset()   # log times in Indian time, shutil
os.chdir("/content/slm-opti"); sys.path.insert(0, "/content/slm-opti"); sys.path.insert(0, "/content/slm-opti/fin")
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-3-4b-it", max_seq_length=4096, load_in_4bit=True, full_finetuning=False)

In [ ]:
# Upload the two files you saved from the first payments run: adapter_model.safetensors and adapter_config.json
from google.colab import files
os.makedirs("/content/pay_adapter", exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, "/content/pay_adapter/" + ("adapter_model.safetensors" if name.endswith(".safetensors") else "adapter_config.json"))
print(sorted(os.listdir("/content/pay_adapter")))
assert {"adapter_model.safetensors", "adapter_config.json"} <= set(os.listdir("/content/pay_adapter")), "Upload both files."

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, "/content/pay_adapter", is_trainable=False)

import stack
def _msgs(messages):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]

@torch.no_grad()
def hf_chat(messages, temperature=0, max_tokens=400):
    inputs = tokenizer.apply_chat_template(_msgs(messages), add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    kw = dict(max_new_tokens=max_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw["temperature"] = temperature
    out = model.generate(**inputs, **kw)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

stack.HF_CHAT = hf_chat

model.eval()
print("Check, 2 + 2 =", hf_chat([{"role": "user", "content": "What is 2 + 2? Reply with the number only."}], max_tokens=5))

In [ ]:
stack.main(["hf:gemma-3-4b-tuned", "--tasks", "tasks_third.json", "--layers", "1234", "--tag", "ft2", "--workers", "1", "--budget", "100000"])

In [ ]:
with zipfile.ZipFile("/content/payments_rule_rerun.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("results/hf-gemma-3-4b-tuned__L1234-ft2.jsonl"):
        z.write(f)
from google.colab import files
files.download("/content/payments_rule_rerun.zip")